In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/Colab Notebooks/NLP

/content/drive/My Drive/Colab Notebooks/NLP


# Importing Libraries

In [ ]:
pip install langchain-text-splitters

In [ ]:
from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names, load_dataset_builder, Dataset
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import numpy as np
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Inspecting Dataset

**Reference**:
_MTS, Department of War UAP Release 1 — structured corpus, 2026. CC-BY-4.0._
This dataset is a structured, machine-readable companion to the source material at [war.gov/UFO/](https://www.war.gov/UFO/).

In [ ]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.77k [00:00<?, ?B/s]

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [ ]:
ds_ufo_builder = load_dataset_builder(ufo_dataset, "pages")
ds_ufo_builder.info.features

In [ ]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

pages.jsonl:   0%|          | 0.00/6.02M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [ ]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

# Check some documents by id

In [ ]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

Let's see if some pages have less than 20 characters.

In [ ]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

total null ids (at least one page): 61
total null pages: 225
problematic ids: ['342-hs1-416511228-319-1-flying-discs-1949', '65-hs1-101634279-100-de-26505', '65-hs1-834228961-62-hq-83894-section-1', '65-hs1-834228961-62-hq-83894-section-10', '65-hs1-834228961-62-hq-83894-section-2', '65-hs1-834228961-62-hq-83894-section-3', '65-hs1-834228961-62-hq-83894-section-4', '65-hs1-834228961-62-hq-83894-section-5', '65-hs1-834228961-62-hq-83894-section-6', '65-hs1-834228961-62-hq-83894-section-7', '65-hs1-834228961-62-hq-83894-section-8', '65-hs1-834228961-62-hq-83894-section-9', '65-hs1-834228961-62-hq-83894-serial-130', '65-hs1-834228961-62-hq-83894-serial-164', '65-hs1-834228961-62-hq-83894-serial-438', 'dow-uap-d10-mission-report-middle-east-may-2022', 'dow-uap-d12-mission-report-iraq-may-2022', 'dow-uap-d25-mission-report-greece-january-2024', 'dow-uap-d27-mission-report-united-arab-emirates-october-2023', 'dow-uap-d28-mission-report-iraq-september-2024', 'dow-uap-d3-mission-report-arabian

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [ ]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only
# Print the single paged documents
# for doc in pages.filter(lambda x: x["document_id"] in single): print(f"DOCUMENT: {doc["document_id"].upper()}\n\n{doc["text"]}\n\n\n\n")

In [ ]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

In [ ]:
filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')

Filter:   0%|          | 0/4239 [00:00<?, ? examples/s]

# Embedding

The embedder is choosen from [here](https://huggingface.co/spaces/mteb/leaderboard)


**[Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) Model Architecture**: <br>
is designed using dual-encoder and cross-encoder architectures the Embedding model processes a single text segment as input, extracting the semantic representation by utilizing the hidden state vector corresponding to the final [EOS] token. ([ref](https://qwen.ai/blog?id=qwen3-embedding))

![Qwen3](images/Qwen3.png)


[article](https://arxiv.org/pdf/2506.05176)
For text embeddings, we utilize LLMs with causal attention, appending an
[EOS] token at the end of the input sequence. The final embedding is derived from the hidden state of the last layer corresponding to this [EOS] token.
To ensure embeddings follow instructions during downstream tasks, we concatenate the instruction and the query into a single input context, while leaving the document unchanged before processing with LLMs. The input format for queries is as follows:
`{Instruction}{Query}<|endoftext|>`

In poche parole: vogliamo tensori che hanno lo stesso numero di token → facciamo padding = aggiungiamo dei [PAD] tokens per riempire lo spazio che manca. Attenzione: il pad lo facciamo a sinistra perché il modello alla fine della fiera si basa sull'ultimo token (EOS) che dovrebbe aver compresso in sè il significato di tutt l'input (l'input è dato da: task-accoppiata a-quesry + documenti), quindi a destra devo lasciare il token [EOS] che prelevo. Questi [PAD] vengono filtrati grazie alla `attention_mask` (1=token vero, 0=è un PAD lascia stare).

In [ ]:
# defining the embedding model and the relative tokenizer
# for the tokenizer we need the left padding
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", cache_dir='models_cache')

# We recommend enabling flash_attention_2 for better acceleration and memory saving.
# model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Per ora stiamo:
- dividendo il testo di ogni pagina in chunk (la divisione avviene con il limite di 512 token e l'overlap di 77 ma è "interna", l'output sono dei chunk di testo, non di token)
- mettendo insieme (come previsto) task-query-chunk_di_testo
- dandoli come input al tokenizer il cui output sono degli effettivi token (paddati)
- dandoli come input al modello (che non worka perché la RAM muore)

Il punto è: a noi ci serve LangChain se vogliamo la divisione con overlap ma ha senso dare chunk come input? o forse sarebbe meglio task-query-**pagina**_di_testo? tanto il limite è di 8192 tokens.

In [ ]:
# function to create the chunked text (input of Qwen3 embedder)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

pages_chunked = []    # list of dictionaries

for page in filterPages:
    page_text = page['text']
    chunk_list = text_splitter.split_text(page_text)

    for i, chunk in enumerate(chunk_list):
        pages_chunked.append({
            'document_id': page['document_id'],
            'page_no': page['page_no'],
            'text': page['text'],
            'chunk_text': chunk
    })

In [ ]:
# just a test: here we lose the metadata
# but we still use LangChain for the overlap thing otherwise we can't do that
chunks = []
for dic in pages_chunked:
  chunks.append(dic['chunk_text'])

In [ ]:
# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# we "merge" in one string instruction and query
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query specifying page and document_id of the passages'

queries = [
    get_detailed_instruct(task, 'what was spotted in the sky for the first time?'),
    get_detailed_instruct(task, 'Have UFOs ever been close to humans (astronauts)?')
]

#documents = chunks
input_texts = queries + chunks

max_length = 512

# Tokenize the input texts
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)
batch_dict.to(model.device)
outputs = model(**batch_dict)
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T)*100
print(scores.tolist())
# [[0.7645568251609802, 0.14142508804798126], [0.13549736142158508, 0.5999549627304077]]


In [ ]:
# bozza

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = [
    get_detailed_instruct(task, 'What is the capital of China?'),
    get_detailed_instruct(task, 'Explain gravity')
]
# No need to add instruction for retrieval documents
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
]
input_texts = queries + documents

max_length = 8192

# Tokenize the input texts
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

In [ ]:
input_texts

['Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:What is the capital of China?',
 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:Explain gravity',
 'The capital of China is Beijing.',
 'Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.']

In [ ]:
batch_dict

{'input_ids': tensor([[   641,   1235,     25,  16246,    264,   3482,   2711,   3239,     11,
          17179,   9760,  46769,    429,   4226,    279,   3239,    198,   2859,
             25,   3838,    374,    279,   6722,    315,   5616,     30, 151643,
         151643, 151643, 151643, 151643],
        [   641,   1235,     25,  16246,    264,   3482,   2711,   3239,     11,
          17179,   9760,  46769,    429,   4226,    279,   3239,    198,   2859,
             25,    840,  20772,  23249, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643],
        [   785,   6722,    315,   5616,    374,  26549,     13, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643],
        [ 38409,    374,    264,   5344,    429,  60091,   1378,  12866,   6974,
           1817,   1008,     13,   1084,   6696,  

---

In [ ]:
Path("models_cache").mkdir(exist_ok=True)
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased", cache_dir="models_cache")
for ID in useIDS:
    vecPrec = []
    for pageText in filterPages.filter(lambda x: x["document_id"]==ID)["text"]:
        toks = tokenizer_bert(pageText, return_attention_mask=False, return_token_type_ids=False, add_special_tokens=False, max_length=int(1e18))["input_ids"]
        toks = vecPrec[-77:] + toks
        vecs = [toks[i:i+512] for i in range(0, len(toks), 435)]
        vecPrec = vecs[-1]

# Join pages of same documents

In [ ]:
# Example of a joined document
docs = pages.filter(lambda x: x["document_id"] == list(useIDS)[0])
text = "\n------------------\n".join(docs["text"])  # The line won't be present in the actual dataset, it's just for showing
print(text)

FEDERAL BUREAU OF INVESTIGATION

Date of entry 10/ /2023

On September 2023, and and FBI Special Agent interviewed via Facetime video. a with sat in on the interview ( and were in at the time of the interview). After being advised of the identity of the interviewing agents and the nature of the interview, provided the following information:

was a in the of He'd been an since and had ten to fifteen hours as a drone pilot.

On September 2023 was at with three other contractors and for LiDAR tests with After receiving a the five of them got into three vehicles with driving the first vehicle and as her passenger.

The three vehicles began driving south around 7:30 am. The sun was in the east with good visibility.

The three vehicles came to a gate that was giving trouble. At about three quarters of the windshield up, saw a linear object with a super bright light on the east side of the object. The light was bright white and bright enough to see bands within the light. The object was metal

In [ ]:
# Create new dataset with joined pages
newDs = {"document_id": [], "text": []}
for ID in useIDS:
    entries = pages.filter(lambda x: x["document_id"] == ID)
    newDs["document_id"].append(ID)
    newDs["text"].append("\n".join(entries["text"]))
newDs = Dataset.from_dict(newDs)

# Retrieval

Basically the rag laboratory, for the time being using only the first doc...

In [ ]:
Path("models_cache").mkdir(exist_ok=True)
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased", cache_dir="models_cache")

In [ ]:
toks = tokenizer_bert(newDs[0]["text"], return_attention_mask=False, return_token_type_ids=False, add_special_tokens=False, max_length=int(1e18))["input_ids"]
vecs = [toks[i:i+512] for i in range(0, len(toks), 435)]

In [ ]:
model_bert = AutoModel.from_pretrained("bert-base-uncased", cache_dir="models_cache")

In [ ]:
lastDs = {"document_id": [], "text": [], "embedding": []}
# perhaps it is possible to batch this process...
for vec in tqdm(vecs):
    hidden = model_bert(input_ids=torch.tensor([vec]), token_type_ids=torch.zeros(1, len(vec), dtype=int), attention_mask=torch.ones(1, len(vec))).last_hidden_state
    embedding = hidden.mean(dim=1).squeeze().detach().cpu().numpy()
    lastDs["text"].append(tokenizer_bert.decode(vec))
    lastDs["embedding"].append(embedding)
lastDs["document_id"] = ['65-hs1-834228961-62-hq-83894-section-9'] * len(lastDs["embedding"])

100%|███████████████████████████████████████████████████████████████████████| 158/158 [01:00<00:00,  2.62it/s]


In [ ]:
query = "What is the saucer's secret?"
enc = tokenizer_bert([query], return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    out = model_bert(**enc)
    embQ = out.last_hidden_state.mean(dim=1).cpu().tolist()[0]
embQCos = np.array(embQ).reshape(1,-1)
embCos = np.array(lastDs["embedding"])
embQCos = embQCos / np.clip(np.linalg.norm(embQCos, axis=1, keepdims=True), 1e-12, None)
embCos = embCos / np.clip(np.linalg.norm(embCos, axis=1, keepdims=True), 1e-12, None)
# cosine sim
embQCos = embQCos @ embCos.T
top_idx = np.argsort(-embQCos[0])[:5]

In [ ]:
for i in top_idx:
    print(lastDs["text"][i], "\n")

. bender of bridgeport, connecticut, who had conducted some research concerning ufos and had allegedly been deterred from his investigation by three unidentified men. according to maney there has been a great deal of speculation regarding the identity of the men who tried to silence bender. he stated that gray barker, editor of " the saucerian bulletin, " had written a book concerning the matter entitled " they knew too much about flying saucers. " 7854 north loma land drive scottsdale, arizona december 7, 1958 federal bureau of investigation washington, d. c. dear sir : i am extremely interested in the " bender affair, " and i will be most grateful if you could help clear up the mystery in this affair. the reason i am writing you is because i have learned recently that a researcher of the flying saucer mystery said the fbi is involved in this affair. a civilian investigating agency was formed in bridgeport, conn., in 1952, to look into the flying saucer mystery by albert k. bender. he

## Iterate over docs

In [ ]:
lastDs = {"document_id": [], "text": [], "embedding": []}
# perhaps it is possible to batch this process...
for doc in newDs:
    toks = tokenizer_bert(doc["text"], return_attention_mask=False, return_token_type_ids=False, add_special_tokens=False, max_length=int(1e18))["input_ids"]
    vecs = [toks[i:i+512] for i in range(0, len(toks), 435)]
    for vec in tqdm(vecs):
        hidden = model_bert(input_ids=torch.tensor([vec]), token_type_ids=torch.zeros(1, len(vec), dtype=int), attention_mask=torch.ones(1, len(vec))).last_hidden_state
        embedding = hidden.mean(dim=1).squeeze().detach().cpu().numpy()
        lastDs["text"].append(tokenizer_bert.decode(vec))
        lastDs["embedding"].append(embedding)
    lastDs["document_id"] += ['65-hs1-834228961-62-hq-83894-section-9'] * len(lastDs["embedding"])

In [ ]:
model_bert = AutoModel.from_pretrained("bert-base-uncased", cache_dir="models_cache")

In [ ]:
query = "What is the saucer's secret?"
enc = tokenizer_bert([query], return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    out = model_bert(**enc)
    embQ = out.last_hidden_state.mean(dim=1).cpu().tolist()[0]
embQCos = np.array(embQ).reshape(1,-1)
embCos = np.array(lastDs["embedding"])
embQCos = embQCos / np.clip(np.linalg.norm(embQCos, axis=1, keepdims=True), 1e-12, None)
embCos = embCos / np.clip(np.linalg.norm(embCos, axis=1, keepdims=True), 1e-12, None)
# cosine sim
embQCos = embQCos @ embCos.T
top_idx = np.argsort(-embQCos[0])[:5]